# Курсовой проект по ТОБД

**Тема:** «Анализ данных о поездках такси для выявления факторов, влияющих на длительность поездок и временных паттернов».

## 2.2–2.3. Загрузка источников и первичная проверка качества

**Источники:** NYC Yellow Taxi (Parquet), Chicago Taxi Trips (XML), Open-Meteo для Нью-Йорка и Чикаго (JSON API), NOAA Storm Events (CSV).


In [1]:
pip install pandas pyarrow requests

In [2]:
import io
import pandas as pd
import requests

---
## 1. Нью-Йорк — NYC Yellow Taxi Trip Records (Parquet)

Официальные данные NYC TLC за январь 2024. Загрузка файла целиком (без выборки записей).

Источник: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [3]:
nyc_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
nyc_df = pd.read_parquet(nyc_url)

In [4]:
nyc_df.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [5]:
nyc_df.shape

(2964624, 19)

In [6]:
nyc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [7]:
nyc_df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,2964624.0,NaN,NaN,NaN,1.754204,1.0,2.0,2.0,2.0,6.0,0.43259
tpep_pickup_datetime,2964624,NaN,NaN,NaN,2024-01-17 00:46:36.431092,2002-12-31 22:59:39,2024-01-09 15:59:19.750000,2024-01-17 10:45:37.500000,2024-01-24 18:23:52.250000,2024-02-01 00:01:15,NaN
tpep_dropoff_datetime,2964624,NaN,NaN,NaN,2024-01-17 01:02:13.208130,2002-12-31 23:05:41,2024-01-09 16:16:23,2024-01-17 11:03:51.500000,2024-01-24 18:40:29,2024-02-02 13:56:52,NaN
passenger_count,2824462.0,NaN,NaN,NaN,1.339281,0.0,1.0,1.0,1.0,9.0,0.850282
trip_distance,2964624.0,NaN,NaN,NaN,3.652169,0.0,1.0,1.68,3.11,312722.3,225.462572
RatecodeID,2824462.0,NaN,NaN,NaN,2.069359,1.0,1.0,1.0,1.0,99.0,9.823219
store_and_fwd_flag,2824462,2,N,2813126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,2964624.0,NaN,NaN,NaN,166.017884,1.0,132.0,162.0,234.0,265.0,63.623914
DOLocationID,2964624.0,NaN,NaN,NaN,165.116712,1.0,114.0,162.0,234.0,265.0,69.31535
payment_type,2964624.0,NaN,NaN,NaN,1.161271,0.0,1.0,1.0,1.0,4.0,0.580869


**Количество пропущенных значений по столбцам:**

In [8]:
nyc_df.isna().sum().to_frame("missing_count")

,missing_count
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,140162
trip_distance,0
RatecodeID,140162
store_and_fwd_flag,140162
PULocationID,0
DOLocationID,0
payment_type,0


**Число полностью совпадающих строк:**

In [10]:
nyc_df.duplicated().sum()

np.int64(0)

---
## 2. Чикаго — Chicago Taxi Trips (2024–), XML

Официальный Socrata XML API, январь 2024 года. Для первичной проверки загружаем не более 10 000 записей, а не весь январь.

Источник: https://data.cityofchicago.org/d/ajtu-isnz



In [11]:
chicago_url = "https://data.cityofchicago.org/resource/ajtu-isnz.xml"
chicago_params = {
    "$where": "trip_start_timestamp >= '2024-01-01T00:00:00' AND trip_start_timestamp < '2024-02-01T00:00:00'",
    "$order": "trip_start_timestamp ASC",
    "$limit": 10000,
}
chicago_response = requests.get(chicago_url, params=chicago_params, timeout=120)
chicago_response.raise_for_status()

Исходный XML сохраняем отдельно, а для табличных проверок читаем его в DataFrame. Структуру XML при этом не меняем.

In [12]:
chicago_xml = chicago_response.content
chicago_df = pd.read_xml(io.BytesIO(chicago_xml), xpath=".//row", parser="etree")

In [13]:
chicago_df.head(5)

,_id,_uuid,_position,_address,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,...,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract
0,row-dvcq.azin~ugd5,00000000-0000-0000-8E68-15F0984BF4EB,0,https://data.cityofchicago.org/resource/ajtu-i...,001b45fb611d0a720696a47b3460731cf3ebe6f9,1b7aef0344f45914e7942efad05553934fe222c880a1cb...,2024-01-01T00:00:00,2024-01-01T00:00:00,856.0,11.00,...,Mobile,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),41.938666,-87.711211,POINT (-87.7112105933 41.9386661962),NaN,NaN
1,row-njyv.74zq-nfvu,00000000-0000-0000-B110-79665CB89BC8,0,https://data.cityofchicago.org/resource/ajtu-i...,01afa254f787921edd7ca37457b848a64c549665,7f9a6d676afea5d94d661cacb40cefdd6b22f626637542...,2024-01-01T00:00:00,2024-01-01T00:30:00,1569.0,10.99,...,Credit Card,Globe Taxi,41.965812,-87.655879,POINT (-87.6558787862 41.96581197),NaN,NaN,None,NaN,NaN
2,row-i45j.ayph-xw8m,00000000-0000-0000-30FC-28B070DEC79C,0,https://data.cityofchicago.org/resource/ajtu-i...,033a57e39e82145683090b93f7f660d9623e28b8,511d5a90fddd5d591ca80c1c2969c190f8ad2997c0b6e9...,2024-01-01T00:00:00,2024-01-01T00:15:00,1587.0,10.93,...,Prcard,5 Star Taxi,41.745758,-87.708366,POINT (-87.7083657043 41.7457577128),41.874005,-87.663518,POINT (-87.6635175498 41.874005383),NaN,NaN
3,row-hhhd_zvit-463k,00000000-0000-0000-DAB6-D191689B43C7,0,https://data.cityofchicago.org/resource/ajtu-i...,054bbff2c120ec8b40dbe90b6b15cf52847d46fd,6c1e4e8e25a1b47575b359c5a0844cf23c50e540a86ecd...,2024-01-01T00:00:00,2024-01-01T00:00:00,392.0,1.03,...,Cash,Flash Cab,41.899602,-87.633308,POINT (-87.6333080367 41.899602111),41.922686,-87.649489,POINT (-87.6494887289 41.9226862843),NaN,NaN
4,row-tb9m.q5cn_gum4,00000000-0000-0000-63AB-35BDA12AEC04,0,https://data.cityofchicago.org/resource/ajtu-i...,0b8d881ee35db48343db0f2a526a040f9ab0f520,07009b8e9b32f36656567f7809b970af38e59dbb58704a...,2024-01-01T00:00:00,2024-01-01T00:15:00,1038.0,9.73,...,Mobile,Flash Cab,41.761578,-87.572782,POINT (-87.5727819867 41.7615779081),41.878866,-87.625192,POINT (-87.6251921424 41.8788655841),NaN,NaN


In [14]:
chicago_df.shape

(10000, 27)

In [15]:
chicago_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   _id                         10000 non-null  object 
 1   _uuid                       10000 non-null  object 
 2   _position                   10000 non-null  int64  
 3   _address                    10000 non-null  object 
 4   trip_id                     10000 non-null  object 
 5   taxi_id                     10000 non-null  object 
 6   trip_start_timestamp        10000 non-null  object 
 7   trip_end_timestamp          10000 non-null  object 
 8   trip_seconds                9999 non-null   float64
 9   trip_miles                  10000 non-null  float64
 10  pickup_community_area       9765 non-null   float64
 11  dropoff_community_area      8891 non-null   float64
 12  fare                        9976 non-null   float64
 13  tips                        9976

In [16]:
chicago_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
_id,10000,10000,row-283i_hg88-hi4j,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
_uuid,10000,10000,00000000-0000-0000-15BA-B2C1D45FB0E5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
_position,10000.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
_address,10000,10000,https://data.cityofchicago.org/resource/ajtu-i...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_id,10000,10000,34ceb5e3daab8eb66b26ca7e9e3cd8bbf0b73e80,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxi_id,10000,1500,abd1ffa32433ceabeb49f4461015b38ddc252847ed3a29...,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_start_timestamp,10000,117,2024-01-01T20:15:00,166,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_end_timestamp,10000,124,2024-01-01T20:45:00,165,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_seconds,9999.0,NaN,NaN,NaN,1075.873487,1325.901794,0.0,521.0,1060.0,1491.0,66326.0
trip_miles,10000.0,NaN,NaN,NaN,7.896223,7.552128,0.0,1.0475,5.9,13.97,147.83


**Количество пропущенных значений:**

In [17]:
chicago_df.isna().sum().to_frame("missing_count")

,missing_count
_id,0
_uuid,0
_position,0
_address,0
trip_id,0
taxi_id,0
trip_start_timestamp,0
trip_end_timestamp,0
trip_seconds,1
trip_miles,0


**Дубликаты по уникальному ID поездки (`trip_id`):**

In [18]:
chicago_df.duplicated(subset=["trip_id"]).sum()

np.int64(0)

**Минимальная очистка.** В отличие от NYC TLC у Chicago Taxi Trips имеется идентификатор `trip_id`. Удаляем повторяющиеся ID; оставляем первый экземпляр записи. После чтения XML убеждаемся, что `trip_seconds` и `trip_miles` числовые: это необходимо для дальнейшего анализа, но не изменяет их значения.

In [19]:
chicago_df = chicago_df.drop_duplicates(subset=["trip_id"])

In [20]:
chicago_df["trip_seconds"] = pd.to_numeric(chicago_df["trip_seconds"], errors="coerce")

In [21]:
chicago_df["trip_miles"] = pd.to_numeric(chicago_df["trip_miles"], errors="coerce")

In [22]:
chicago_df = chicago_df.dropna(subset=["trip_id", "trip_start_timestamp", "trip_seconds", "trip_miles"])

In [23]:
chicago_df.shape

(9999, 27)

---
## 3. Нью-Йорк — Open-Meteo Historical Weather API (JSON)

Исторические почасовые данные за январь 2024. Координаты — приблизительный центр Манхэттена, а не погода на каждом маршруте. Временная зона — `America/New_York`.

Источник: https://open-meteo.com/en/docs/historical-weather-api

In [24]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_nyc_params = {
    "latitude": 40.758,
    "longitude": -73.9855,
    "start_date": "2024-01-01",
    "end_date": "2024-01-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,weather_code",
    "timezone": "America/New_York",
}
weather_nyc_response = requests.get(weather_url, params=weather_nyc_params, timeout=120)
weather_nyc_response.raise_for_status()

In [25]:
weather_nyc_json = weather_nyc_response.json()
weather_nyc_df = pd.DataFrame(weather_nyc_json["hourly"])

In [26]:
weather_nyc_df.head(5)

,time,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,weather_code
0,2024-01-01T00:00,2.1,67,0.0,0.0,0.0,7.7,3
1,2024-01-01T01:00,1.8,71,0.0,0.0,0.0,9.0,3
2,2024-01-01T02:00,2.4,71,0.0,0.0,0.0,10.9,3
3,2024-01-01T03:00,2.4,73,0.0,0.0,0.0,10.1,3
4,2024-01-01T04:00,2.2,75,0.0,0.0,0.0,8.1,1


In [27]:
weather_nyc_df.shape

(744, 8)

In [28]:
weather_nyc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  744 non-null    object 
 1   temperature_2m        744 non-null    float64
 2   relative_humidity_2m  744 non-null    int64  
 3   precipitation         744 non-null    float64
 4   rain                  744 non-null    float64
 5   snowfall              744 non-null    float64
 6   wind_speed_10m        744 non-null    float64
 7   weather_code          744 non-null    int64  
dtypes: float64(5), int64(2), object(1)
memory usage: 46.6+ KB


In [29]:
weather_nyc_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
time,744,744,2024-01-31T23:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature_2m,744.0,NaN,NaN,NaN,0.512769,4.706207,-9.5,-2.825,0.6,3.5,13.7
relative_humidity_2m,744.0,NaN,NaN,NaN,71.346774,18.502938,28.0,56.75,73.0,88.0,100.0
precipitation,744.0,NaN,NaN,NaN,0.209946,0.878153,0.0,0.0,0.0,0.0,11.9
rain,744.0,NaN,NaN,NaN,0.197177,0.87786,0.0,0.0,0.0,0.0,11.9
snowfall,744.0,NaN,NaN,NaN,0.009032,0.045478,0.0,0.0,0.0,0.0,0.49
wind_speed_10m,744.0,NaN,NaN,NaN,14.078763,7.815471,0.7,8.1,12.5,19.1,55.0
weather_code,744.0,NaN,NaN,NaN,15.365591,24.517101,0.0,0.0,3.0,3.0,73.0


**Количество пропусков:**

In [30]:
weather_nyc_df.isna().sum().to_frame("missing_count")

,missing_count
time,0
temperature_2m,0
relative_humidity_2m,0
precipitation,0
rain,0
snowfall,0
wind_speed_10m,0
weather_code,0


**Дублирующиеся временные метки:**

In [31]:
weather_nyc_df.duplicated(subset=["time"]).sum()

np.int64(0)

---
## 4. Чикаго — Open-Meteo Historical Weather API (JSON)

Тот же набор почасовых параметров и период, но координаты Чикаго и временная зона `America/Chicago`.

Источник: https://open-meteo.com/en/docs/historical-weather-api

In [32]:
weather_chicago_params = {
    "latitude": 41.8781,
    "longitude": -87.6298,
    "start_date": "2024-01-01",
    "end_date": "2024-01-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,weather_code",
    "timezone": "America/Chicago",
}
weather_chicago_response = requests.get(weather_url, params=weather_chicago_params, timeout=120)
weather_chicago_response.raise_for_status()

In [33]:
weather_chicago_json = weather_chicago_response.json()
weather_chicago_df = pd.DataFrame(weather_chicago_json["hourly"])

In [34]:
weather_chicago_df.head(5)

,time,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,weather_code
0,2024-01-01T00:00,0.6,85,0.1,0.0,0.07,30.4,71
1,2024-01-01T01:00,0.4,85,0.1,0.0,0.07,30.4,71
2,2024-01-01T02:00,0.5,86,0.1,0.0,0.07,30.1,71
3,2024-01-01T03:00,0.8,85,0.1,0.0,0.07,27.8,71
4,2024-01-01T04:00,0.6,85,0.0,0.0,0.00,25.4,3


In [35]:
weather_chicago_df.shape

(744, 8)

In [36]:
weather_chicago_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  744 non-null    object 
 1   temperature_2m        744 non-null    float64
 2   relative_humidity_2m  744 non-null    int64  
 3   precipitation         744 non-null    float64
 4   rain                  744 non-null    float64
 5   snowfall              744 non-null    float64
 6   wind_speed_10m        744 non-null    float64
 7   weather_code          744 non-null    int64  
dtypes: float64(5), int64(2), object(1)
memory usage: 46.6+ KB


In [37]:
weather_chicago_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
time,744,744,2024-01-31T23:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature_2m,744.0,NaN,NaN,NaN,-3.62379,7.330881,-24.9,-6.525,-0.55,1.225,7.2
relative_humidity_2m,744.0,NaN,NaN,NaN,80.331989,13.958552,40.0,70.0,83.0,92.0,100.0
precipitation,744.0,NaN,NaN,NaN,0.146237,0.400453,0.0,0.0,0.0,0.1,3.6
rain,744.0,NaN,NaN,NaN,0.085618,0.295812,0.0,0.0,0.0,0.0,2.6
snowfall,744.0,NaN,NaN,NaN,0.042809,0.169817,0.0,0.0,0.0,0.0,2.52
wind_speed_10m,744.0,NaN,NaN,NaN,18.417876,8.20082,0.5,11.9,17.65,24.2,45.1
weather_code,744.0,NaN,NaN,NaN,19.783602,28.562513,0.0,2.0,3.0,51.0,75.0


**Количество пропусков:**

In [38]:
weather_chicago_df.isna().sum().to_frame("missing_count")

,missing_count
time,0
temperature_2m,0
relative_humidity_2m,0
precipitation,0
rain,0
snowfall,0
wind_speed_10m,0
weather_code,0


**Дублирующиеся временные метки:**

In [39]:
weather_chicago_df.duplicated(subset=["time"]).sum()

np.int64(0)

---
## 5. NOAA Storm Events Database (CSV).

Официальный сжатый CSV NOAA за 2024 год. Для сопоставимости с текущим этапом после чтения выделяем события января 2024 года в штатах New York и Illinois.

Источник: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/



In [40]:
noaa_url = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2024_c20260728.csv.gz"
noaa_all_df = pd.read_csv(noaa_url, compression="gzip", low_memory=False)

**Выбираем данные по двум штатам за январь 2024.** Не считаем их городскими наблюдениями.

In [41]:
noaa_df = noaa_all_df.loc[
    noaa_all_df["STATE"].isin(["NEW YORK", "ILLINOIS"])
    & (noaa_all_df["MONTH_NAME"] == "January")
].copy()

In [42]:
noaa_df.head(5)

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
314,202401,13,500,202401,13,600,187986,1154650,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A strong area of low pressure shifted northeas...,A measured high wind gust of 60 mph occurred a...,CSV
317,202401,14,1600,202401,14,1700,188155,1155609,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"On January 14, 2024, a strong pressure gradien...",A measured high wind gust of 58 mph occurred a...,CSV
649,202401,10,18,202401,10,48,187413,1154988,NEW YORK,36,...,0.8,SE,ARLYN OAKS,40.6678,-73.4442,40.6727,-73.4424,A large low pressure system impacted the area ...,Block Blvd closed in East Massapequa due to fl...,CSV
909,202401,6,1800,202401,7,1300,187076,1148160,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low pressure pushed northeast from the Tenness...,Snowfall totals varied across Orange County fr...,CSV
910,202401,6,1800,202401,7,1300,187076,1148168,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low pressure pushed northeast from the Tenness...,Heavy snow was reported via Cocorahs observers...,CSV


In [43]:
noaa_df.shape

(666, 51)

In [44]:
noaa_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 666 entries, 314 to 15041
Data columns (total 51 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   BEGIN_YEARMONTH     666 non-null    int64  
 1   BEGIN_DAY           666 non-null    int64  
 2   BEGIN_TIME          666 non-null    int64  
 3   END_YEARMONTH       666 non-null    int64  
 4   END_DAY             666 non-null    int64  
 5   END_TIME            666 non-null    int64  
 6   EPISODE_ID          666 non-null    int64  
 7   EVENT_ID            666 non-null    int64  
 8   STATE               666 non-null    object 
 9   STATE_FIPS          666 non-null    int64  
 10  YEAR                666 non-null    int64  
 11  MONTH_NAME          666 non-null    object 
 12  EVENT_TYPE          666 non-null    object 
 13  CZ_TYPE             666 non-null    object 
 14  CZ_FIPS             666 non-null    int64  
 15  CZ_NAME             666 non-null    object 
 16  WFO      

In [52]:
noaa_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
BEGIN_YEARMONTH,666.0,NaN,NaN,NaN,202401.0,0.0,202401.0,202401.0,202401.0,202401.0,202401.0
BEGIN_DAY,666.0,NaN,NaN,NaN,14.405405,5.904063,1.0,10.0,14.0,16.0,30.0
BEGIN_TIME,666.0,NaN,NaN,NaN,1061.459459,693.425633,0.0,500.0,1112.5,1600.0,2356.0
END_YEARMONTH,666.0,NaN,NaN,NaN,202401.0,0.0,202401.0,202401.0,202401.0,202401.0,202401.0
END_DAY,666.0,NaN,NaN,NaN,15.400901,5.840481,7.0,12.0,14.0,17.0,31.0
END_TIME,666.0,NaN,NaN,NaN,1137.656156,637.276017,0.0,800.0,1200.0,1512.5,2359.0
EPISODE_ID,666.0,NaN,NaN,NaN,187998.412913,582.421889,186853.0,187334.0,188115.0,188572.0,188859.0
EVENT_ID,666.0,NaN,NaN,NaN,1155225.108108,4408.175648,1146891.0,1150525.25,1155395.5,1159184.75,1161994.0
STATE,666,2,ILLINOIS,353,NaN,NaN,NaN,NaN,NaN,NaN,NaN
STATE_FIPS,666.0,NaN,NaN,NaN,25.929429,9.489978,17.0,17.0,17.0,36.0,36.0


**Количество пропусков по столбцам:**

In [46]:
noaa_df.isna().sum().to_frame("missing_count")

,missing_count
BEGIN_YEARMONTH,0
BEGIN_DAY,0
BEGIN_TIME,0
END_YEARMONTH,0
END_DAY,0
END_TIME,0
EPISODE_ID,0
EVENT_ID,0
STATE,0
STATE_FIPS,0


**Повторяющиеся идентификаторы погодных событий:**

In [48]:
noaa_df.duplicated(subset=["EVENT_ID"]).sum()

np.int64(0)

---
## Итог текущего этапа

После выполнения всех ячеек готовы пять табличных объектов:

- `nyc_df` — поездки Нью-Йорка;
- `chicago_df` — поездки Чикаго;
- `weather_nyc_df` — почасовая погода Нью-Йорка;
- `weather_chicago_df` — почасовая погода Чикаго;
- `noaa_df` — январские погодные события в штатах New York и Illinois.

